In [1]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import (
    EsmForMaskedLM, 
    EsmConfig,
    PretrainedConfig, 
    EsmTokenizer, 
    DataCollatorForLanguageModeling, 
    Trainer
)

from tokenizers import Tokenizer
import torch
import torch.nn.functional as F
from torch import Tensor, nn

from sklearn.linear_model import LinearRegression

import einops
import yaml
import sys
import json
import functools
import os
import shutil

import numpy as np
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model
from datasets import Dataset, load_dataset
import math
from tqdm import tqdm

from matplotlib import pyplot as plt

from jaxtyping import Bool, Float, Int
from plotly.subplots import make_subplots
import plotly.express as px
from plotly_utils import (
    imshow,
    line,
    bar
)

import circuitsvis as cv
from IPython.display import display, HTML
from IPython import get_ipython
ip = get_ipython()
if not ip.extension_manager.loaded:
    ip.extension_manager.load('autoreload')
    %autoreload 2

In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences, get_protein_sequence
from interp_utils import get_hooked_state_dict, get_hooked_esm_config, get_logits_hooked_esm, get_fairesm_state_dict

In [3]:
from covfit_stuff.config import Config, ModelConfig
from covfit_stuff.esm_regression import load_model_for_inference, get_model_predictions, EsmForRegression
import tempfile

# Load CovFit model

In [4]:
TOK_DIR = "./covfit_stuff/Tokenizer"
CONF_DIR = "./covfit_stuff/Config"
TASK_IDS_FILE = "./covfit_stuff/task_id_dict.pt"
FOLD_ID = 0
N_TARGETS = 1565
MODEL_PATH = f"./covfit_stuff/models/covfit_model_20241007_{FOLD_ID}.ckpt"

In [5]:
# MODEL_PATH = "TheSatoLab-UTokyo/CoVFit"
# FOLD_IDS_TO_USE = [0]
# TARGET_FOLD_ID = 0
# OUTPUT_PREFIX = "inference_results"

model_name = "facebook/esm2_t33_650M_UR50D"
device = "cuda"
CONTEXT_LEN = 1024
torch.autograd.grad_mode.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [6]:
def get_model(
    TOK_DIR = "./covfit_stuff/Tokenizer",
    CONF_DIR = "./covfit_stuff/Config",
    TASK_IDS_FILE = "./covfit_stuff/task_id_dict.pt",
    FOLD_ID = 0,
    N_TARGETS = 1565,
    MODEL_PATH = f"./covfit_stuff/models/covfit_model_20241007_{FOLD_ID}.ckpt",
    device=device
):
    esm_config = EsmConfig.from_pretrained(CONF_DIR)
    model = EsmForRegression(esm_config, N_TARGETS).to(device)

    lora_config = LoraConfig(
        task_type="SEQ_CLS",
        r=8,
        lora_alpha=16,
        target_modules=["key", "query", "value","dense"],
        lora_dropout=0.05,
        bias="lora_only",
        modules_to_save=["regressor"]
    )
    esm_fine_tuned = get_peft_model(model, lora_config)
    state_dict = torch.load(MODEL_PATH, map_location=device)
    
    # keys_to_remove = []
    # for key in state_dict.keys():
    #     if 'contact_head' in key:
    #         keys_to_remove.append(key)
    
    # for key in keys_to_remove:
    #     del state_dict[key]

    wrong_keys = [key for key in state_dict.keys() if key not in esm_fine_tuned.state_dict().keys()]
    key_list = list(state_dict.keys())
    for key in key_list:
        if key in wrong_keys:
            correct_key = key.rsplit('.',1)[0]+'.base_layer.'+key.rsplit('.',1)[1]
            state_dict[correct_key] = state_dict.pop(key)

    del state_dict["base_model.model.esm.embeddings.position_embeddings.base_layer.weight"]
    
    esm_fine_tuned.load_state_dict(state_dict)
    esm_fine_tuned = esm_fine_tuned.merge_and_unload()
    esm_fine_tuned.eval()
    esm_fine_tuned.esm.embeddings.token_dropout = False

    return esm_fine_tuned

In [7]:
esm_fine_tuned = get_model()
# save_state_dict = False
# if save_state_dict:
#     torch.save(esm_fine_tuned.state_dict(), "./covfit_stuff/models/covfit_esm_statedict.pt")

In [8]:
esm_fine_tuned = esm_fine_tuned.to(device)
esm_fine_tuned = esm_fine_tuned.eval()

In [9]:
esm_config = esm_fine_tuned.config
esm_config.token_dropout = False
esm_config.model_name = model_name
REPO_ID = esm_config.model_name
original_task_id_infos = torch.load("./covfit_stuff/task_id_dict.pt", map_location=device)

In [10]:
tokenizer_config = {}
special_tokens_map_file = "./covfit_stuff/Tokenizer/special_tokens_map.json"
tokenizer_config["vocab_file"] = "./covfit_stuff/Tokenizer/vocab.txt"
tokenizer_config["model_max_length"] = CONTEXT_LEN

with open("./covfit_stuff/Tokenizer/special_tokens_map.json", "r") as f:
    tokenizer_config = {**tokenizer_config, **(json.load(f))}

In [11]:
with open(tokenizer_config["vocab_file"], "r") as f:
    f_data = f.read().split("\n")
    aa_to_toks_map = {i:f_data[i] for i in range(len(f_data))}
    aa_to_toks_map_rev = {aa_to_toks_map[k]:k for k in aa_to_toks_map.keys()}

In [12]:
tokenizer = EsmTokenizer(**tokenizer_config)

hooked_esm_config = get_hooked_esm_config(esm_config, context_len=CONTEXT_LEN, use_hook_tokens=True)
hooked_esm = HookedTransformer(hooked_esm_config)
print(hooked_esm.load_state_dict(get_hooked_state_dict(esm_fine_tuned.state_dict(), hooked_esm_config)))

<All keys matched successfully>


In [13]:
# clean up memory
torch.cuda.empty_cache()

# Load Data

In [14]:
def tokenizer_for_map(seq, seq_key="input_ids", tokenizer=tokenizer): #Tokenizer and params including special_tokens_mask required for MLM
    return tokenizer(
        seq[seq_key],
        return_tensors="pt", 
        return_special_tokens_mask=True,
        truncation=True,
        padding="max_length",
        max_length=300,
    )

In [15]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
pathogens = list(config["pathogens"].keys())
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,return_tensors='pt',mlm_probability=0.15)

In [16]:
MAX_LEN=1024
pathogen_suffixes = ["africa", "asia", "europe", "north_america", "oceania", "south_america"]
d_out_vocab = esm_fine_tuned.regressor[3].weight.size(0)
pathogen_name = "sars_cov_2_spike"
protein_coords = config["pathogens"][f"{pathogen_name}_africa"]["protein_coords"]

In [17]:
"""
uniq_seqs - seqs used in training
seq_names - names of ALL sequences
all_seqs - ALL sequences
seq_idxs - map from seq_names to uniq_seqs, i.e. seq_names[i] is for uniq_seqs[seq_idxs[i]]
"""

all_seqs = []
seq_names = []
seq_idxs = []
all_uniq_seqs = []

for suff in pathogen_suffixes:
    fasta_file = f"../../data/pathogen/{pathogen_name}_{suff}/alignment.fasta"
    data = load_sequences(fasta_file)
    sequence_names, sequences = list(zip(*list(data.items())))
    sequences = [get_protein_sequence(x, protein_coords) for x in sequences]

    keep_idx = [i for i,x in enumerate(sequences) if len(x.replace("-","")) > (CONTEXT_LEN // 5) * 4]
    sequences = [sequences[i] for i in keep_idx]
    sequence_names = [sequence_names[i] for i in keep_idx]
    
    uniq_seqs_suff, unique_inv_idx  = np.unique(sequences, return_inverse=True) # For the purpose of eval, I only care about unique sequences 

    all_seqs.extend(sequences)
    seq_names.extend(sequence_names)
    seq_idxs.extend(unique_inv_idx + len(all_uniq_seqs))
    all_uniq_seqs.extend(uniq_seqs_suff)

all_uniq_seqs, unique_inv_idx  = np.unique(all_uniq_seqs, return_inverse=True) # For the purpose of eval, I only care about unique sequences 
seq_idxs = [unique_inv_idx[idx] for idx in seq_idxs]
all_uniq_seqs = list(all_uniq_seqs)

# identical code to how it's compute_node_embeddings.py
tok_output = tokenizer(all_uniq_seqs, return_tensors="pt", return_special_tokens_mask=True, truncation=True, padding="max_length", max_length=MAX_LEN)
tok_seqs = tok_output.input_ids.to(device)
tok_masks = tok_output.attention_mask.to(device)

print(pathogen_name)
print(f"Number unique sequences: {len(all_uniq_seqs)}")
print(tok_seqs.shape)

sars_cov_2_spike
Number unique sequences: 4404
torch.Size([4404, 1024])


In [18]:
# add padding mask to model
def add_perma_hooks_to_mask_pad_tokens(
    model: HookedTransformer, pad_token: int
) -> HookedTransformer:
    # Hook which operates on the tokens, and stores a mask where tokens equal [pad]
    def cache_padding_tokens_mask(tokens: Float[Tensor, "batch seq"], hook: HookPoint) -> None:
        hook.ctx["padding_tokens_mask"] = einops.rearrange(tokens == pad_token, "b sK -> b 1 1 sK")

    # Apply masking, by referencing the mask stored in the `hook_tokens` hook context
    def apply_padding_tokens_mask(
        attn_scores: Float[Tensor, "batch head seq_Q seq_K"],
        hook: HookPoint,
    ) -> None:
        attn_scores.masked_fill_(model.hook_dict["hook_tokens"].ctx["padding_tokens_mask"], -1e5)
        if hook.layer() == model.cfg.n_layers - 1:
            del model.hook_dict["hook_tokens"].ctx["padding_tokens_mask"]

    # Add these hooks as permanent hooks (i.e. they aren't removed after functions like run_with_hooks)
    for name, hook in model.hook_dict.items():
        if name == "hook_tokens":
            hook.add_perma_hook(cache_padding_tokens_mask)
        elif name.endswith("attn_scores"):
            hook.add_perma_hook(apply_padding_tokens_mask)

    return model


hooked_esm.reset_hooks(including_permanent=True)
hooked_esm = add_perma_hooks_to_mask_pad_tokens(hooked_esm, 1)

In [19]:
component_name_map = dict()
for l in range(esm_config.num_hidden_layers + 1):
    if l < esm_config.num_hidden_layers:
        component_name_map[l] = f"blocks.{l}.hook_resid_pre"
    
    # final layer
    elif l == esm_config.num_hidden_layers:
        component_name_map[l] = f"unembed.hook_in"

In [20]:
def get_logit_hooked(output: Float[Tensor, "batch pos d_model"], tok_id):
    logits = get_logits_hooked_esm(output[:,0,:], esm_fine_tuned.regressor)[:,tok_id]
    torch.cuda.empty_cache()
    return logits

In [21]:
def get_rev_names(id_seq):
    """
    Given seq x in all_uniq_seqs, get the corresponding name(s) of sequences that have the same spike protein
    """
    if type(id_seq) == int:
        id_seq = [id_seq]

    rev_name_dict = {}
    for id_s in id_seq:
        name_idx = np.argwhere(np.array(seq_idxs) == id_s)[:,0]
        rev_name_dict[id_s] = [seq_names[x] for x in name_idx]
    return rev_name_dict

#### Given some sequence with name seq_names[i], its corresponding sequence is all_uniq_seqs[seq_idxs[i]]
#### Given some sequence with index uniq_seqs[i], its corresponding name(s) is get_rev_names(i)

In [22]:
logit_id = original_task_id_infos["fitness_USA"]

## Cross-Layer Transcoder experiments

## Curating annotated dataset

In [45]:
N = 455
L455_seqs = [x for x in all_uniq_seqs if x[N-1] == "L"]
L455_idx = [i for i,x in enumerate(all_uniq_seqs) if x[N-1] == "L"]
synthetic_seqs = [x[:N-1] + "F" + x[N:] for x in L455_seqs]

In [24]:
wt_idx = [i for i,x in enumerate(seq_names) if "wuhan" in x.lower()][0]
wt_seq = np.array(list(all_uniq_seqs[seq_idxs[wt_idx]]))[None, :]
np_seqs = np.array([list(x) for x in [*all_uniq_seqs, *synthetic_seqs]])

In [25]:
total_cls_data_num = np_seqs.shape[0]

In [26]:
seq_muts = [] # List[List[str]] where each element is the number of muts from element i for wt (WuHan)
num_muts = []

diffs = np.argwhere(np_seqs != wt_seq)
for s in range(np_seqs.shape[0]):
    diff_aa_idx = diffs[diffs[:,0] == s][:,1] #mutations are 1-indexed
    num_muts.append(diff_aa_idx.shape[0])
    
    seq_muts.append([f"Spike-{wt_seq[0, x]}{x+1}{np_seqs[s, x]}" if np_seqs[s, x] != "-" else f"del Spike-{wt_seq[0, x]}{x+1}" for x in diff_aa_idx])

seq_muts = [str(x)[1:-1].replace("'","") for x in seq_muts]

In [27]:
equiv_class_seq_names = list(get_rev_names(list(range(len(all_uniq_seqs)))).values())
equiv_class_seq_names = [
    [x for x in elem if "NODE" not in x] if len([x for x in elem if "NODE" not in x]) > 0  else ["unlabeled internal node(s)"] for elem in equiv_class_seq_names 
]
equiv_class_seq_names = equiv_class_seq_names + [[f"(synthetic L455F) {x}" for x in equiv_class_seq_names[i]] for i in L455_idx]
equiv_class_seq_names = [str(x)[1:-1].replace("'","") for x in equiv_class_seq_names]

In [31]:
clt_data_path = "../../data"
clt_data_dict = pd.DataFrame.from_dict({
    "id": equiv_class_seq_names,
    "Mutations": seq_muts,
    "Num mutations": num_muts,
    "sequence": [*all_uniq_seqs, *synthetic_seqs]
})
clt_data_dict.to_parquet(os.path.join(clt_data_path, "cls_data.parquet"))

## Testing CLT

In [23]:
sys.path.append("../../ProtoMech/")
sys.path.append("../../ProtoMech/training_transcoder")
sys.path.append("../../ProtoMech/training")
import argparse
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from data_module import SequenceDataModule
from plt_module import PLTLightningModule
import random

In [24]:
parser = argparse.ArgumentParser()
# Path params
parser.add_argument("--data-dir", type=str, required=True, help="Path to .a2m or .parquet file")
parser.add_argument("--esm2-weight", type=str, required=True, help="Path to ESM2 weights .pt file")
parser.add_argument("--output-dir", type=str, default="results_plt", help="Directory for checkpoints/logs")

# Model params
parser.add_argument("--num-layers", type=int, default=6, help="Total layers in pLM")
parser.add_argument("--d-model", type=int, default=320)
parser.add_argument("--d-hidden", type=int, default=3200, help="Latent dim per layer")

# Training params
parser.add_argument("--batch-size", type=int, default=32)
parser.add_argument("--lr", type=float, default=2e-4)
parser.add_argument("--k", type=int, default=16)
parser.add_argument("--auxk", type=int, default=32)
parser.add_argument("--dead-steps-threshold", type=int, default=10000)
parser.add_argument("--max-epochs", type=int, default=1)
parser.add_argument("--num-devices", type=int, default=1)
parser.add_argument("--wandb-project", type=str, default="ESM-PLT")

_StoreAction(option_strings=['--wandb-project'], dest='wandb_project', nargs=None, const=None, default='ESM-PLT', type=<class 'str'>, choices=None, required=False, help=None, metavar=None)

In [25]:
clt_data_path = "../../data"

In [26]:
arg_dict = {
    "--data-dir": os.path.join(clt_data_path, "cls_data.parquet"),
    "--esm2-weight":"./covfit_stuff/models/covfit_esm_statedict.pt",
    "--output-dir": "./covfit_stuff/CLT_test",
    "--num-layers": esm_config.num_hidden_layers,
    "--d-model": esm_config.hidden_size, # d_model of ESM
    "--d-hidden": 16 * esm_config.hidden_size, # latent dim of CLT 
    "--batch-size": 32, 
}
args = parser.parse_args([str(x) for (k,v) in arg_dict.items() for x in (k,v)])

In [27]:
run_name = f"PLT_L{args.num_layers}_H{args.d_hidden}"
run_output_dir = os.path.join(args.output_dir, run_name)
os.makedirs(run_output_dir, exist_ok=True)

In [58]:
model = PLTLightningModule(args).to(device)
print("instatiated model!")
fairesm_state_dict = get_fairesm_state_dict(esm_fine_tuned.state_dict(), esm_config)
print("created state dict!")

del model.esm_model.lm_head
del model.esm_model.contact_head
print(model.esm_model.load_state_dict(fairesm_state_dict))
model.esm_model.lm_head = lambda x: x

instatiated model!
created state dict!
<All keys matched successfully>


In [59]:
data_module = SequenceDataModule(args.data_dir, args.batch_size)

In [60]:
checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join(run_output_dir, "checkpoints"),
    filename="plt-{step}-{val/loss:.2f}",
    save_top_k=2,
    monitor="val/loss", 
    mode="min",
    save_last=True
)

In [61]:
trainer = pl.Trainer(
    max_epochs=args.max_epochs,
    accelerator="gpu",
    devices=args.num_devices,
    # logger=wandb_logger,
    callbacks=[checkpoint_callback],
    gradient_clip_val=1.0,
    val_check_interval=1000, 
    limit_val_batches=10,
    log_every_n_steps=1 
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/averma2/miniforge3/envs/exploratory_env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [62]:
trainer.validate(model, data_module)

You are using a CUDA device ('NVIDIA H100 NVL') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Data Loaded: Train 6382, Val 797, Test 799


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val/avg_nmse        │    1.0026566982269287     │
│         val/loss          │    33.087669372558594     │
│     val/nmse_layer_0      │     1.002311110496521     │
│     val/nmse_layer_1      │    1.0041955709457397     │
│     val/nmse_layer_10     │     1.003393292427063     │
│     val/nmse_layer_11     │     1.003151535987854     │
│     val/nmse_layer_12     │     1.002202033996582     │
│     val/nmse_layer_13     │    1.0010234117507935     │
│     val/nmse_layer_14     │    1.0004931688308716     │
│     val/nmse_layer_15     │    1.0013072490692139     │
│     val/nmse_layer_16     │    1.0008634328842163     │
│     val/nmse_layer_17     │    1.0005853176116943     │
│     val/nmse_layer_18     │     1.000645399093628     │
│     val/nmse_layer_19     │    1.0006470680236816     │
│     val/nmse_layer_2      │    1.0068846940994263     │
│     val/nmse_layer_20     │    1.0009373426437378     │
│     val/nmse_layer_21     │    1.0010324716567993     │
│     val/nmse_layer_22     │    1.0009398460388184     │
│     val/nmse_layer_23     │    1.0007638931274414     │
│     val/nmse_layer_24     │    1.0009843111038208     │
│     val/nmse_layer_25     │     1.000750184059143     │
│     val/nmse_layer_26     │    1.0011152029037476     │
│     val/nmse_layer_27     │    1.0009262561798096     │
│     val/nmse_layer_28     │    1.0008007287979126     │
│     val/nmse_layer_29     │    1.0006530284881592     │
│     val/nmse_layer_3      │     1.008023977279663     │
│     val/nmse_layer_30     │     1.000234603881836     │
│     val/nmse_layer_31     │    1.0002564191818237     │
│     val/nmse_layer_32     │     1.000154733657837     │
│     val/nmse_layer_4      │    1.0089877843856812     │
│     val/nmse_layer_5      │    1.0078954696655273     │
│     val/nmse_layer_6      │    1.0082343816757202     │
│     val/nmse_layer_7      │     1.006523609161377     │
│     val/nmse_layer_8      │    1.0060312747955322     │
│     val/nmse_layer_9      │     1.004719853401184     │
└───────────────────────────┴───────────────────────────┘

[{'val/nmse_layer_0': 1.002311110496521,
  'val/nmse_layer_1': 1.0041955709457397,
  'val/nmse_layer_2': 1.0068846940994263,
  'val/nmse_layer_3': 1.008023977279663,
  'val/nmse_layer_4': 1.0089877843856812,
  'val/nmse_layer_5': 1.0078954696655273,
  'val/nmse_layer_6': 1.0082343816757202,
  'val/nmse_layer_7': 1.006523609161377,
  'val/nmse_layer_8': 1.0060312747955322,
  'val/nmse_layer_9': 1.004719853401184,
  'val/nmse_layer_10': 1.003393292427063,
  'val/nmse_layer_11': 1.003151535987854,
  'val/nmse_layer_12': 1.002202033996582,
  'val/nmse_layer_13': 1.0010234117507935,
  'val/nmse_layer_14': 1.0004931688308716,
  'val/nmse_layer_15': 1.0013072490692139,
  'val/nmse_layer_16': 1.0008634328842163,
  'val/nmse_layer_17': 1.0005853176116943,
  'val/nmse_layer_18': 1.000645399093628,
  'val/nmse_layer_19': 1.0006470680236816,
  'val/nmse_layer_20': 1.0009373426437378,
  'val/nmse_layer_21': 1.0010324716567993,
  'val/nmse_layer_22': 1.0009398460388184,
  'val/nmse_layer_23': 1.0007

In [ ]:
trainer.fit(model, data_module)